In [153]:
#Import des librairies
import pandas as pd
import numpy as np
import os

# Mise en forme du fichier de référence pour les pays

In [154]:
ref_country = pd.read_csv("data/all.csv", delimiter=",")
ref_country.head()

,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
0,Afghanistan,AF,AFG,4,ISO 3166-2:AF,Asia,Southern Asia,NaN,142.0,34.0,NaN
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,Northern Europe,NaN,150.0,154.0,NaN
2,Albania,AL,ALB,8,ISO 3166-2:AL,Europe,Southern Europe,NaN,150.0,39.0,NaN
3,Algeria,DZ,DZA,12,ISO 3166-2:DZ,Africa,Northern Africa,NaN,2.0,15.0,NaN
4,American Samoa,AS,ASM,16,ISO 3166-2:AS,Oceania,Polynesia,NaN,9.0,61.0,NaN


In [155]:
#Renommer les colonnes
ref_country = ref_country[['name', 'alpha-3', 'region', 'sub-region']]
ref_country = ref_country.rename(columns={
    'name': 'country_name',
    'alpha-3': 'country_code',
    'region': 'continent',
    'sub-region': 'sub_region'
})
ref_country.head()

,country_name,country_code,continent,sub_region
0,Afghanistan,AFG,Asia,Southern Asia
1,Åland Islands,ALA,Europe,Northern Europe
2,Albania,ALB,Europe,Southern Europe
3,Algeria,DZA,Africa,Northern Africa
4,American Samoa,ASM,Oceania,Polynesia


In [156]:
# Analyse de doublons
print(ref_country['country_code'].duplicated().sum())  # doit être 0
print(ref_country.shape)  # ~250 lignes

0
(249, 4)


# Mise en forme du fichier pour le PIB

GDP MKTP

In [157]:
gdp_total = pd.read_csv("data/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_126992.csv", skiprows=4)
gdp_total.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,3.092428e+09,3.276188e+09,3.346623e+09,2.471419e+09,2.880903e+09,3.324034e+09,3.834730e+09,4.265651e+09,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP (current US$),NY.GDP.MKTP.CD,2.420569e+10,2.495889e+10,2.707323e+10,3.176914e+10,3.027955e+10,3.380618e+10,...,9.780765e+11,1.020956e+12,1.018715e+12,9.386076e+11,1.114145e+12,1.228968e+12,1.179359e+12,1.242694e+12,NaN,NaN
2,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,1.875346e+10,1.805322e+10,1.879944e+10,1.995593e+10,1.426000e+10,1.449724e+10,1.715223e+10,NaN,NaN,NaN
3,Africa Western and Central,AFW,GDP (current US$),NY.GDP.MKTP.CD,1.190481e+10,1.270772e+10,1.363059e+10,1.446891e+10,1.580356e+10,1.692088e+10,...,6.940513e+11,7.778403e+11,1.026996e+12,9.637847e+11,1.026651e+12,1.063649e+12,9.382384e+11,7.363850e+11,NaN,NaN
4,Angola,AGO,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,8.437694e+10,8.951279e+10,8.073443e+10,5.885246e+10,7.955954e+10,1.312122e+11,1.071677e+11,1.009989e+11,NaN,NaN


In [158]:
#Filtrer sur les colonnes voulues
years_cols = [str(y) for y in range(2015, 2025)]
gdp_total = gdp_total[['Country Name', 'Country Code'] + years_cols]

#Pivot pour avoir les données sous la bonne forme
gdp_total = gdp_total.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='year',
    value_name='gdp_total_usd'
)

In [159]:
#Renommer les colonnes
gdp_total['year'] = gdp_total['year'].astype(int)
gdp_total = gdp_total.rename(columns={
    'Country Name': 'country_name',
    'Country Code': 'country_code'
})

In [160]:
codes_valides = ref_country['country_code'].unique()
gdp_total = gdp_total[gdp_total['country_code'].isin(codes_valides)]

In [161]:
# Lignes avec valeurs nulles
print(f"Lignes avec NaN : {gdp_total['gdp_total_usd'].isna().sum()}")
gdp_total = gdp_total.dropna(subset=['gdp_total_usd'])

Lignes avec NaN : 89


In [162]:
print(gdp_total.shape)
print(gdp_total.head())
print(f"Pays uniques : {gdp_total['country_code'].nunique()}")
print(f"Années : {gdp_total['year'].min()} - {gdp_total['year'].max()}")

(2061, 4)
  country_name country_code  year  gdp_total_usd
0        Aruba          ABW  2015   2.962907e+09
2  Afghanistan          AFG  2015   1.913422e+10
4       Angola          AGO  2015   1.025431e+11
5      Albania          ALB  2015   1.147017e+10
6      Andorra          AND  2015   2.789883e+09
Pays uniques : 211
Années : 2015 - 2024


GDP PCAP

In [163]:
gdp_pcap = pd.read_csv("data\API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_121708.csv",delimiter=",",skiprows=4)

#Filtrer sur les colonnes voulues
years_cols = [str(y) for y in range(2015, 2026)]
gdp_pcap = gdp_pcap[['Country Name', 'Country Code'] + years_cols]

#Pivot pour avoir les colonnes dans le bon sens
gdp_pcap = gdp_pcap.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='year',
    value_name='gdp_per_capita_ppp'
)

#Renommer les colonnes
gdp_pcap['year'] = gdp_pcap['year'].astype(int)
gdp_pcap = gdp_pcap.rename(columns={
    'Country Name': 'country_name',
    'Country Code': 'country_code'
})

codes_valides = ref_country['country_code'].unique()
gdp_pcap = gdp_pcap[gdp_pcap['country_code'].isin(codes_valides)]

print(gdp_pcap.shape)
gdp_pcap.head()

(2365, 4)


,country_name,country_code,year,gdp_per_capita_ppp
0,Aruba,ABW,2015,35972.859662
2,Afghanistan,AFG,2015,2284.075848
4,Angola,AGO,2015,8067.485847
5,Albania,ALB,2015,12386.592267
6,Andorra,AND,2015,50733.239001


# Mise en forme du fichier pour le chômage

In [164]:
#Fonction pour simplifier le traitement
def clean_wb_indicator(filepath, value_name, codes_valides, year_start=2015, year_end=2025):
    """
    Charge et nettoie un fichier Banque Mondiale (format wide multi-années).
    
    Parameters
    ----------
    filepath : str
        Chemin du fichier CSV brut (format API_*.csv)
    value_name : str
        Nom de la colonne valeur après transformation (ex: 'unemployment_rate')
    codes_valides : iterable
        Codes ISO 3 lettres des pays à conserver (filtre les agrégats régionaux)
    year_start, year_end : int
        Bornes de la période à conserver (incluses)
    
    Returns
    -------
    pd.DataFrame
        Format long avec colonnes : country_name, country_code, year, <value_name>
    """
    df = pd.read_csv(filepath, skiprows=4)
    years_cols = [str(y) for y in range(year_start, year_end + 1)]
    df = df[['Country Name', 'Country Code'] + years_cols]
    df = df.melt(
        id_vars=['Country Name', 'Country Code'],
        var_name='year',
        value_name=value_name
    )
    df['year'] = df['year'].astype(int)
    df = df.rename(columns={
        'Country Name': 'country_name',
        'Country Code': 'country_code'
    })
    df = df[df['country_code'].isin(codes_valides)]
    return df

In [165]:
codes_valides = ref_country['country_code'].unique()

#Application de la fonction sur unemployment
unemployment = clean_wb_indicator(
    "data/API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_115692.csv",
    'unemployment_rate',
    codes_valides
)

print(unemployment.shape)
unemployment.head()

(2365, 4)


,country_name,country_code,year,unemployment_rate
0,Aruba,ABW,2015,NaN
2,Afghanistan,AFG,2015,9.032
4,Angola,AGO,2015,16.472
5,Albania,ALB,2015,17.193
6,Andorra,AND,2015,NaN


In [166]:
print(f"Lignes vides : {unemployment['unemployment_rate'].isna().sum()} sur {len(unemployment)}")
print(f"Pays sans aucune donnée : {unemployment.groupby('country_code')['unemployment_rate'].apply(lambda x: x.isna().all()).sum()}")

Lignes vides : 333 sur 2365
Pays sans aucune donnée : 29


# Mise en forme du fichier pour l'espérance de vie

In [167]:
life_exp = clean_wb_indicator(
    "data/API_SP.DYN.LE00.IN_DS2_en_csv_v2_121663.csv",
    'life_expectancy',
    codes_valides
)

print(life_exp.shape)
life_exp.head()

(2365, 4)


,country_name,country_code,year,life_expectancy
0,Aruba,ABW,2015,75.405
2,Afghanistan,AFG,2015,62.270
4,Angola,AGO,2015,61.042
5,Albania,ALB,2015,78.358
6,Andorra,AND,2015,84.532


# Jointure entre les différents indicateurs

In [168]:
wb = (gdp_total
      .merge(gdp_pcap, on=['country_code', 'country_name', 'year'], how='outer')
      .merge(unemployment, on=['country_code', 'country_name', 'year'], how='outer')
      .merge(life_exp, on=['country_code', 'country_name', 'year'], how='outer'))

print(wb.shape)
wb.head()

(2365, 7)


,country_name,country_code,year,gdp_total_usd,gdp_per_capita_ppp,unemployment_rate,life_expectancy
0,Aruba,ABW,2015,2.962907e+09,35972.859662,NaN,75.405
1,Aruba,ABW,2016,2.983637e+09,36117.535262,NaN,75.540
2,Aruba,ABW,2017,3.092428e+09,37524.914920,NaN,75.620
3,Aruba,ABW,2018,3.276188e+09,39287.059713,NaN,75.880
4,Aruba,ABW,2019,3.346623e+09,38543.907294,NaN,76.019


# Mise en forme du fichier VDEM

In [169]:
# Charger UNIQUEMENT les colonnes utiles avec usecols
cols_a_garder = [
    'country_name',
    'country_text_id',  # code ISO 3 lettres = ta clé de jointure
    'year',
    'v2x_polyarchy',    # Electoral Democracy Index
    'v2x_libdem',       # Liberal Democracy Index
    'v2x_partipdem',    # Participatory Democracy
    'v2x_egaldem',      # Egalitarian Democracy
    'v2x_corr'          # Political Corruption Index
]

vdem = pd.read_csv("data/V-Dem-CY-Core-v16.csv", usecols=cols_a_garder)
print(vdem.shape)
vdem.head()

(28092, 8)


,country_name,country_text_id,year,v2x_polyarchy,v2x_libdem,v2x_partipdem,v2x_egaldem,v2x_corr
0,Mexico,MEX,1789,0.027,0.044,0.006,NaN,0.68
1,Mexico,MEX,1790,0.027,0.044,0.006,NaN,0.68
2,Mexico,MEX,1791,0.027,0.044,0.006,NaN,0.68
3,Mexico,MEX,1792,0.027,0.044,0.006,NaN,0.68
4,Mexico,MEX,1793,0.027,0.044,0.006,NaN,0.68


In [170]:
# Renommer pour la clarté
vdem = vdem.rename(columns={
    'country_text_id': 'country_code',
    'v2x_polyarchy': 'democratie_electorale',
    'v2x_libdem': 'democratie_liberale',
    'v2x_partipdem': 'democratie_participative',
    'v2x_egaldem': 'democratie_egalitaire',
    'v2x_corr': 'corruption_politique'
})

# Filtrer la période
vdem = vdem[(vdem['year'] >= 2015) & (vdem['year'] <= 2025)]

# Filtrer pour ne garder que les pays présents dans ton référentiel ISO
codes_valides = ref_country['country_code'].unique()
vdem = vdem[vdem['country_code'].isin(codes_valides)]

# Vérification
print(vdem.shape)
print(f"Pays uniques : {vdem['country_code'].nunique()}")
print(f"Années : {vdem['year'].min()} - {vdem['year'].max()}")
vdem.head()

(1925, 8)
Pays uniques : 175
Années : 2015 - 2025


,country_name,country_code,year,democratie_electorale,democratie_liberale,democratie_participative,democratie_egalitaire,corruption_politique
226,Mexico,MEX,2015,0.643,0.433,0.399,0.329,0.724
227,Mexico,MEX,2016,0.643,0.434,0.399,0.328,0.724
228,Mexico,MEX,2017,0.638,0.440,0.407,0.333,0.682
229,Mexico,MEX,2018,0.675,0.452,0.428,0.372,0.652
230,Mexico,MEX,2019,0.684,0.428,0.420,0.396,0.607


In [171]:
print("Pays présents dans WHR + WB + V-Dem (intersection) :")
iso_wb = set(wb['country_code'].unique())
iso_vdem = set(vdem['country_code'].unique())

commun = iso_wb & iso_vdem
print(f"  Banque Mondiale + V-Dem : {len(commun)} pays")

Pays présents dans WHR + WB + V-Dem (intersection) :
  Banque Mondiale + V-Dem : 174 pays


# Mise en forme du fichier pour le bonheur

In [172]:
happiness = pd.read_csv("data/world_happiness_report_2005_2025.csv")

print(happiness.shape)

print(happiness.columns.tolist())
happiness.head()

(2116, 13)
['year', 'rank_in_year', 'country', 'happiness_score', 'lower_whisker', 'upper_whisker', 'explained_log_gdp_per_capita', 'explained_social_support', 'explained_healthy_life_expectancy', 'explained_freedom', 'explained_generosity', 'explained_corruption', 'dystopia_plus_residual']


,year,rank_in_year,country,happiness_score,lower_whisker,upper_whisker,explained_log_gdp_per_capita,explained_social_support,explained_healthy_life_expectancy,explained_freedom,explained_generosity,explained_corruption,dystopia_plus_residual
0,2011,1,Denmark,7.856,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011,2,Finland,7.579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011,3,Norway,7.524,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2011,4,Netherlands,7.512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2011,5,Canada,7.499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [173]:
# Renommer en plus clair
happiness = happiness.rename(columns={
    'country': 'country_name',
    'happiness_score': 'score_bonheur',
    'explained_log_gdp_per_capita': 'fact_pib',
    'explained_social_support': 'fact_soutien_social',
    'explained_healthy_life_expectancy': 'fact_esperance_vie',
    'explained_freedom': 'fact_liberte',
    'explained_generosity': 'fact_generosite',
    'explained_corruption': 'fact_corruption',
    'dystopia_plus_residual': 'dystopia_residual'
})

# Filtrer 2015-2025
happiness = happiness[(happiness['year'] >= 2015) & (happiness['year'] <= 2025)]

# Ajouter le code ISO via jointure avec ref_country
happiness = happiness.merge(
    ref_country[['country_code', 'country_name']],
    on='country_name',
    how='left'
)

# Vérifier les pays sans correspondance ISO (point critique)
sans_iso = happiness[happiness['country_code'].isna()]['country_name'].unique()
print(f"Pays WHR sans correspondance ISO ({len(sans_iso)}) :")
print(sans_iso)

Pays WHR sans correspondance ISO (20) :
['Netherlands' 'United States' 'United Kingdom' 'Taiwan Province of China'
 'Venezuela' 'Republic of Moldova' 'Republic of Korea' 'Bolivia'
 'North Cyprus' 'Hong Kong SAR of China' 'Kosovo' 'Somaliland Region'
 'Lao PDR' 'Iran' 'State of Palestine' 'DR Congo' 'Côte d’Ivoire'
 'Tanzania' 'Syria' 'Swaziland']


In [174]:
pays_mapping = {
    # Variantes de format simple
    'United States': 'United States of America',
    'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland',
    
    # Noms officiels ISO avec virgule
    'Netherlands': 'Netherlands, Kingdom of the',
    'Venezuela': 'Venezuela, Bolivarian Republic of',
    'Bolivia': 'Bolivia, Plurinational State of',
    'Iran': 'Iran, Islamic Republic of',
    'Republic of Korea': 'Korea, Republic of',
    'Republic of Moldova': 'Moldova, Republic of',
    'State of Palestine': 'Palestine, State of',
    'Tanzania': 'Tanzania, United Republic of',
    'Taiwan Province of China': 'Taiwan, Province of China',
    
    # Noms officiels ISO sans virgule
    'Lao PDR': "Lao People's Democratic Republic",
    'Syria': 'Syrian Arab Republic',
    'DR Congo': 'Congo, Democratic Republic of the',
    'Hong Kong SAR of China': 'Hong Kong',
    
    # Renommages historiques
    'Swaziland': 'Eswatini',
    
    # Apostrophe typographique → apostrophe droite
    'Côte d\u2019Ivoire': "Côte d'Ivoire",
}

happiness['country_name'] = happiness['country_name'].replace(pays_mapping)

# Refaire la jointure
happiness = happiness.drop(columns=['country_code'], errors='ignore')
happiness = happiness.merge(
    ref_country[['country_code', 'country_name']],
    on='country_name',
    how='left'
)

# Vérif
sans_iso = happiness[happiness['country_code'].isna()]['country_name'].unique()
print(f"Restants sans ISO : {len(sans_iso)} → {sans_iso}")

Restants sans ISO : 3 → ['North Cyprus' 'Kosovo' 'Somaliland Region']


# Jointure

In [175]:
wb = (gdp_total
      .merge(gdp_pcap, on=['country_code', 'country_name', 'year'], how='outer')
      .merge(unemployment, on=['country_code', 'country_name', 'year'], how='outer')
      .merge(life_exp, on=['country_code', 'country_name', 'year'], how='outer'))

print(wb.shape)
wb.head()

(2365, 7)


,country_name,country_code,year,gdp_total_usd,gdp_per_capita_ppp,unemployment_rate,life_expectancy
0,Aruba,ABW,2015,2.962907e+09,35972.859662,NaN,75.405
1,Aruba,ABW,2016,2.983637e+09,36117.535262,NaN,75.540
2,Aruba,ABW,2017,3.092428e+09,37524.914920,NaN,75.620
3,Aruba,ABW,2018,3.276188e+09,39287.059713,NaN,75.880
4,Aruba,ABW,2019,3.346623e+09,38543.907294,NaN,76.019


In [176]:
#Analyse des différents fichiers
iso_happiness = set(happiness['country_code'].dropna().unique())
iso_wb = set(wb['country_code'].unique())
iso_vdem = set(vdem['country_code'].unique())

commun = iso_happiness & iso_wb & iso_vdem
print(f"Pays présents dans les 3 sources : {len(commun)}")
print(f"Dans WHR uniquement : {len(iso_happiness - iso_wb - iso_vdem)}")
print(f"Dans WB mais pas WHR : {len(iso_wb - iso_happiness)}")

Pays présents dans les 3 sources : 158
Dans WHR uniquement : 0
Dans WB mais pas WHR : 55


In [177]:
print("Le pays uniquement dans WHR :")
print(iso_happiness - iso_wb - iso_vdem)

Le pays uniquement dans WHR :
set()


Couverture des sources

WHR (bonheur) : 159 pays
Banque Mondiale : 213 pays
V-Dem (démocratie) : ~180 pays
Périmètre cœur (3 sources) : 158 pays

In [178]:
os.makedirs("data/clean", exist_ok=True)

# Dimension pays (référentiel)
ref_country.to_csv("data/clean/dim_pays.csv", index=False, encoding='utf-8-sig')

# Faits
happiness.to_csv("data/clean/fact_happiness.csv", index=False, encoding='utf-8-sig')
wb.to_csv("data/clean/fact_economy.csv", index=False, encoding='utf-8-sig')
vdem.to_csv("data/clean/fact_democracy.csv", index=False, encoding='utf-8-sig')

print("Export terminé")

Export terminé
